# DB2 Warehouse Connectivity - Interaction

This notebook demonstrates how to connect to the Db2 warehouse instance, some basic functionality, as well as running queries in the database via python.

## Import Modules

In [2]:
import os
import ibm_db
import pandas as pd
import pyarrow.flight as flight
import itc_utils.flight_service as itcfs
from ibm_watson_studio_lib import access_project_or_space
from project_lib import Project
from pyspark.sql import SparkSession
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)

#initiate spark session
sparkSession = SparkSession(spark).builder.getOrCreate()

## Interact with Db2 Warehouse

### Write to database

In [4]:
# specify ddl sql query to run
static_statement = """
CREATE TABLE "ERMINF_RAW"."ERMH_YEARLY_KWH_FOLIO"  (
		  "ACCT_COMPANY" CHAR(1 OCTETS) NOT NULL , 
		  "ACCT_FOLIO" CHAR(8 OCTETS) NOT NULL , 
		  "FY10" DOUBLE , 
		  "FY9" DOUBLE , 
		  "FY8" DOUBLE , 
		  "FY7" DOUBLE , 
		  "FY6" DOUBLE , 
		  "FY5" DOUBLE , 
		  "FY4" DOUBLE , 
		  "FY3" DOUBLE , 
		  "FY2" DOUBLE , 
		  "FY1" DOUBLE , 
		  "FY0" DOUBLE );
"""

In [5]:
# define connection to use and run query
nb_data_request = {
    'connection_name': """con-db2wh-1""",
    'interaction_properties': {
        'static_statement': static_statement,
        'write_mode': 'static_statement'
    },
}

flightClient = itcfs.get_flight_client()
flight_request = itcfs.get_data_request(nb_data_request=nb_data_request)
flight_request['context'] = 'target'

flight_cmd = itcfs.get_flight_cmd(data_request=flight_request)

action = flight.Action('setup_phase',flight_cmd.encode('utf-8'))
gen = flightClient.do_action(action)

RuntimeError: No asset named "con-db2wh-1" of type "connection".

### Read from database

In [6]:
# query to retrieve table names from erminf_src schema
table_names = """
SELECT NAME FROM sysibm.systables WHERE CREATOR = 'ERMINF_SRC' AND TYPE = 'T';
"""

In [ ]:
# establish and run the request
nb_data_request = {
    'connection_name': """con-db2wh-1""",
    'interaction_properties': {
        'select_statement': table_names
    }
}
flight_request = itcfs.get_data_request(nb_data_request=nb_data_request)

# get result in a spark dataframe
tables = sparkSession.read.format("com.ibm.connect.spark.flight") \
    .option("flight.location", itcfs.get_flight_service_url()) \
    .option("flight.command", itcfs.get_flight_cmd(data_request=flight_request)) \
    .option("flight.authToken", itcfs.get_bearer_token()) \
    .load()

In [ ]:
# create a subset of the dataframe 
tables_subset = tables.where((tables.NAME == 'ERMH_CSKATKOT_ALL') | (tables.NAME == 'ERMH_YEARLY_KWH_FOLIO') | (tables.NAME == 'ERMH_REGISTER_KOT'))

In [ ]:
# collect the subset of erminf_src table names in a list
tables_filtered = [data[0] for data in tables_subset.select('NAME').collect()]